In [ ]:
%pip install ipywidgets

In [ ]:
import io, warnings
import numpy as np
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
from scipy.ndimage import zoom as nd_zoom, rotate as nd_rotate, gaussian_filter
from ipywidgets import IntSlider, FloatSlider, Dropdown, VBox, HBox, Layout, Output
from IPython.display import display, Image, clear_output
warnings.filterwarnings('ignore')

# Shared layout and label-width settings for all interactive sliders
slider_layout       = Layout(width="380px")
slider_layout_wide  = Layout(width="420px")
slider_layout_wider = Layout(width="440px")
label_style = {"description_width": "120px"}


# ---------------------------------------------------------------------------
# Test-image helpers
# ---------------------------------------------------------------------------

def make_letter_image(text, size=64):
    """Render a single character as a binary 2-D image of the given pixel size.

    Steps:
      1. Draw the character with matplotlib on a small canvas.
      2. Average the R, G, B channels to obtain a grayscale array.
      3. Normalise to [0, 1], invert so letter pixels = 1, background = 0.
      4. Flip vertically for standard mathematical orientation.
      5. Zoom (resize) to the requested pixel size.
    """
    fig, ax = plt.subplots(figsize=(2, 2))
    ax.axis('off')
    fig.text(0.23, 0.26, text, fontsize=100)
    fig.canvas.draw()
    raw_buffer = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    canvas_width, canvas_height = fig.canvas.get_width_height()
    grayscale = raw_buffer.reshape(canvas_height, canvas_width, 4)[:, :, :3].mean(2)
    plt.close(fig)
    grayscale = (grayscale - grayscale.min()) / (grayscale.max() - grayscale.min() + 1e-10)
    binary_image = (~grayscale.astype(bool))[::-1].astype(float)
    return nd_zoom(binary_image, size / binary_image.shape[0])


def make_circle_image(size=64, radius_fraction=0.35):
    """Create a filled disc image — a featureless reference with no built-in orientation bias."""
    y, x = np.mgrid[0:size, 0:size].astype(float)
    return ((x - size / 2) ** 2 + (y - size / 2) ** 2 < (size * radius_fraction) ** 2).astype(float)


def low_pass_filter(image, sigma=3.0):
    """Smooth an image with a Gaussian of the given standard deviation (in pixels).

    Low-pass filtered images are used as initial references: they lack fine detail
    and therefore introduce less reference bias than an unblurred structure.
    """
    return gaussian_filter(image, sigma=sigma)


def show_image(image, ax=None, title='', vmin=None, vmax=None, cmap='gray'):
    """Display a 2-D image in a matplotlib axes, hiding axis ticks."""
    if ax is None:
        _, ax = plt.subplots()
    ax.imshow(image, cmap=cmap, origin='lower',
              vmin=vmin if vmin is not None else image.min(),
              vmax=vmax if vmax is not None else image.max())
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=9)


def figure_to_image(fig):
    """Render a matplotlib figure to a PNG and return it as an IPython Image for notebook display."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    plt.close(fig)
    return Image(data=buf.read())


# ---------------------------------------------------------------------------
# Image formation and noise simulation
# ---------------------------------------------------------------------------

def simulate_single_image(clean_image, rotation_angle_deg, noise_sigma=1, rng=None):
    """Simulate one cryo-EM particle image.

    Model: X = R(θ) · A + σ · G
      A          — clean 2-D structure
      R(θ)       — rotation by θ degrees
      σ · G      — additive Gaussian noise with standard deviation σ

    Returns the noisy, rotated image.
    """
    rng = rng or np.random.default_rng()
    rotated = nd_rotate(clean_image, rotation_angle_deg, reshape=False)
    return rotated + noise_sigma * rng.standard_normal(clean_image.shape)


def simulate_particle_stack(clean_image, noise_sigma, num_images, seed=42):
    """Simulate a stack of num_images particle images at random orientations.

    Each image is rotated by a uniformly random angle in [0°, 360°) and
    then corrupted with Gaussian noise of standard deviation noise_sigma.

    Returns
    -------
    true_angles : (N,) array of ground-truth rotation angles in degrees
    image_stack : (N, H, W) array of simulated images
    """
    rng = np.random.default_rng(seed)
    true_angles = rng.uniform(0, 360, num_images)
    image_stack = np.array([
        simulate_single_image(clean_image, angle, noise_sigma, rng)
        for angle in true_angles
    ])
    return true_angles, image_stack


# ---------------------------------------------------------------------------
# Reference generation and template matching
# ---------------------------------------------------------------------------

def build_reference_stack(reference_image, angular_step_deg=5):
    """Pre-rotate a reference image at every candidate angle.

    The alignment algorithm compares each particle image against all rotated
    copies of the reference to find the best-matching orientation.

    Returns
    -------
    candidate_angles : (M,) array of angles sampled every angular_step_deg degrees
    rotated_references : (M, H, W) array of rotated reference images
    """
    candidate_angles = np.arange(0, 360, angular_step_deg)
    rotated_references = np.array([
        nd_rotate(reference_image, angle, reshape=False)
        for angle in candidate_angles
    ])
    return candidate_angles, rotated_references


def compute_cross_correlation(particle_image, reference_stack):
    """Compute the normalised dot-product between a particle image and every reference in the stack.

    A higher score means the particle image and reference look more alike.
    This is a simple (non-normalised) cross-correlation; real SPA software
    uses normalised cross-correlation or the Fourier-space equivalent.

    Returns a 1-D array of scores, one per reference in the stack.
    """
    return (particle_image[None] * reference_stack).mean(axis=(1, 2))


def align_particle_stack(image_stack, reference_stack, candidate_angles):
    """Align every particle image to the best-matching candidate angle.

    For each image, score it against every rotated reference and record the
    angle that gives the highest cross-correlation.

    Returns
    -------
    estimated_angles : (N,) array of estimated rotation angles in degrees
    """
    return np.array([
        candidate_angles[int(compute_cross_correlation(img, reference_stack).argmax())]
        for img in image_stack
    ])


def average_aligned_images(image_stack, estimated_angles):
    """Reconstruct a 2-D structure by back-rotating and averaging all particle images.

    Each image is rotated by −θ_i (undoing the estimated rotation) and then
    averaged. With many images the noise averages to zero, revealing the structure.

    Returns the averaged (reconstructed) image.
    """
    reconstruction = np.zeros_like(image_stack[0])
    for angle, image in zip(estimated_angles, image_stack):
        reconstruction += nd_rotate(image, -angle, reshape=False)
    return reconstruction / len(image_stack)


# ---------------------------------------------------------------------------
# Translation handling
# ---------------------------------------------------------------------------

def translate_image(image, shift_x, shift_y):
    """Shift an image by (shift_x, shift_y) pixels using circular boundary conditions."""
    return np.roll(np.roll(image, shift_y, axis=0), shift_x, axis=1)


def simulate_stack_with_translation(clean_image, noise_sigma, num_images, max_shift=10, seed=42):
    """Simulate particle images with unknown rotation AND translation.

    Real particle images are not centred: the particle can sit anywhere in
    the box. This function generates images where each particle is also
    shifted by a random integer offset (dx, dy) before noise is added.

    Returns
    -------
    image_stack  : (N, H, W) noisy images
    true_angles  : (N,) ground-truth rotation angles
    true_dx      : (N,) ground-truth x shifts in pixels
    true_dy      : (N,) ground-truth y shifts in pixels
    """
    rng = np.random.default_rng(seed)
    true_angles = rng.uniform(0, 360, num_images)
    true_dx = rng.integers(-max_shift, max_shift + 1, num_images)
    true_dy = rng.integers(-max_shift, max_shift + 1, num_images)
    image_stack = []
    for angle, dx, dy in zip(true_angles, true_dx, true_dy):
        rotated = nd_rotate(clean_image, angle, reshape=False)
        shifted = translate_image(rotated, dx, dy)
        image_stack.append(shifted + noise_sigma * rng.standard_normal(clean_image.shape))
    return np.array(image_stack), true_angles, true_dx, true_dy


def build_rotation_translation_reference_stack(reference_image, angular_step_deg=10,
                                                max_shift=8, shift_step=4):
    """Build a reference stack that spans all combinations of rotation and translation.

    For each (angle, dx, dy) combination, store the rotated-and-shifted reference.
    This allows joint alignment over rotation AND translation by simple template matching.

    Returns
    -------
    reference_stack : (M, H, W) array of all rotated+shifted references
    angle_list      : (M,) rotation angle for each reference
    dx_list         : (M,) x-shift for each reference
    dy_list         : (M,) y-shift for each reference
    """
    candidate_angles = np.arange(0, 360, angular_step_deg)
    shift_values     = np.arange(-max_shift, max_shift + 1, shift_step)
    reference_stack, angle_list, dx_list, dy_list = [], [], [], []
    for angle in candidate_angles:
        rotated = nd_rotate(reference_image, angle, reshape=False)
        for dx in shift_values:
            for dy in shift_values:
                reference_stack.append(translate_image(rotated, dx, dy))
                angle_list.append(angle)
                dx_list.append(dx)
                dy_list.append(dy)
    return (np.array(reference_stack), np.array(angle_list),
            np.array(dx_list), np.array(dy_list))


def align_with_translation(image_stack, reference_stack, angle_list, dx_list, dy_list):
    """Find the best (angle, dx, dy) for each particle image via matrix cross-correlation.

    Reshaping to 2-D (images × pixels) allows all dot products to be computed at
    once with a single matrix multiplication — much faster than a Python loop.

    Returns
    -------
    estimated_angles : (N,) best-fit rotation angles
    estimated_dx     : (N,) best-fit x shifts
    estimated_dy     : (N,) best-fit y shifts
    """
    num_images, height, width = image_stack.shape
    num_references = reference_stack.shape[0]
    # Flatten spatial dimensions for fast matrix multiply
    images_2d     = image_stack.reshape(num_images, height * width).astype(float)
    references_2d = reference_stack.reshape(num_references, height * width).astype(float)
    # Cross-correlation matrix: shape (num_images, num_references)
    correlation_matrix = (images_2d @ references_2d.T) / (height * width)
    best_reference_indices = correlation_matrix.argmax(1)
    return (angle_list[best_reference_indices],
            dx_list[best_reference_indices],
            dy_list[best_reference_indices])


def reconstruct_with_translation(image_stack, estimated_angles, estimated_dx, estimated_dy):
    """Back-rotate and back-shift every image, then average to form the reconstruction."""
    reconstruction = np.zeros_like(image_stack[0])
    for image, angle, dx, dy in zip(image_stack, estimated_angles, estimated_dx, estimated_dy):
        # Undo translation first, then undo rotation
        unshifted = translate_image(image, -dx, -dy)
        reconstruction += nd_rotate(unshifted, -angle, reshape=False)
    return reconstruction / len(image_stack)


# ---------------------------------------------------------------------------
# Maximum-likelihood (soft) alignment
# ---------------------------------------------------------------------------

def log_likelihood(particle_image, reference_image, noise_sigma):
    """Compute the log-likelihood that particle_image was generated from reference_image.

    Assuming Gaussian noise: log P(X | A, σ) = −||X − A||² / (2σ²) + const.
    The constant cancels when comparing likelihoods, so we return only the
    variable part (up to an irrelevant additive constant).
    """
    return -0.5 * np.sum((particle_image - reference_image) ** 2) / noise_sigma ** 2


def ml_reconstruct(image_stack, reference_stack, candidate_angles, noise_sigma=2.0):
    """Soft (maximum-likelihood) reconstruction.

    Instead of committing to a single best angle for each image (hard assignment),
    we compute a probability weight for every candidate angle:

        w_ij  ∝  exp(log P(X_i | R(θ_j) A))

    The reconstruction is then a weighted average over all angles:

        A ← (1/N) Σ_i  Σ_j  w_ij · R(−θ_j) X_i

    At high SNR the weights peak sharply at the true angle (recovering hard assignment).
    At low SNR they spread across many angles, averaging out alignment uncertainty.
    """
    reconstruction = np.zeros_like(image_stack[0])
    for particle_image in image_stack:
        # Log-likelihoods for each candidate reference
        log_likelihoods = np.array([
            log_likelihood(particle_image, ref, noise_sigma)
            for ref in reference_stack
        ])
        # Shift for numerical stability before exponentiating
        log_likelihoods -= log_likelihoods.max()
        weights = np.exp(log_likelihoods)
        weights /= weights.sum()   # normalise to a probability distribution
        # Weighted sum: each back-rotated image contributes according to its weight
        for weight, angle in zip(weights, candidate_angles):
            reconstruction += weight * nd_rotate(particle_image, -angle, reshape=False)
    return reconstruction / len(image_stack)


# ---------------------------------------------------------------------------
# Pre-compute test images and reference stacks used across multiple widgets.
# Computing these once at setup time keeps the interactive widgets responsive.
# ---------------------------------------------------------------------------

# The ground-truth 2-D structure used throughout the practical
test_image_q = make_letter_image('Q', size=128)

# A library of reference images the student can choose from
reference_images = {
    'Q (oracle)':        test_image_q,
    'Blurred Q (σ=5)':   low_pass_filter(test_image_q, sigma=5),
    'Circle':            make_circle_image(128),
    'A (wrong)':         make_letter_image('A', 128),
    'Blurred A (σ=5)':   low_pass_filter(make_letter_image('A', 128), sigma=5),
    'P (wrong)':         make_letter_image('P', 128),
    'Blurred P (σ=5)':   low_pass_filter(make_letter_image('P', 128), sigma=5),
    'O (wrong)':         make_letter_image('O', 128),
    'Blurred O (σ=5)':   low_pass_filter(make_letter_image('O', 128), sigma=5),
}

# Candidate angles sampled every 5° — used for the main alignment widget
candidate_angles_5deg = np.arange(0, 360, 5)

# Pre-rotated reference stacks for each reference image
reference_stacks_5deg = {
    name: np.array([nd_rotate(ref, a, reshape=False) for a in candidate_angles_5deg])
    for name, ref in reference_images.items()
}

# A fixed example dataset: 200 images at σ=2 used for the alignment demo
true_angles_example, noisy_images_example = simulate_particle_stack(test_image_q, 2.0, 200, seed=42)
blurred_reference_example = low_pass_filter(test_image_q, sigma=5)
candidate_angles_example, reference_stack_example = build_reference_stack(blurred_reference_example, angular_step_deg=5)
estimated_angles_example = align_particle_stack(noisy_images_example, reference_stack_example, candidate_angles_example)

# Pre-computed stacks for the classification demo (letters Q, P, R)
class_letters = ['Q', 'P', 'R']
class_images  = {letter: make_letter_image(letter, 128) for letter in class_letters}
class_candidate_angles = np.arange(0, 360, 10)
class_reference_stacks = {
    letter: np.array([nd_rotate(class_images[letter], a, reshape=False) for a in class_candidate_angles])
    for letter in class_letters
}

# Pre-computed stacks for the maximum-likelihood demo
ml_reference_images = {
    'Blurred Q':          low_pass_filter(test_image_q, sigma=3),
    'Circle (wrong)':     make_circle_image(128),
    'Letter A (wrong)':   make_letter_image('A', 128),
}
ml_candidate_angles = np.arange(0, 360, 10)
ml_reference_stacks = {
    name: np.array([nd_rotate(ref, a, reshape=False) for a in ml_candidate_angles])
    for name, ref in ml_reference_images.items()
}
# Noisy image sets at three noise levels (σ = 1.5, 3.0, 6.0) for the ML comparison
ml_noisy_images = {
    sigma: simulate_particle_stack(test_image_q, sigma, 10, seed=42)[1]
    for sigma in [1.5, 3.0, 6.0]
}


# ===========================================================================
# Widget factories — each function creates and displays one interactive panel
# ===========================================================================

def show_test_image():
    """Display the ground-truth 2-D structure used throughout the practical."""
    fig, ax = plt.subplots(figsize=(3, 3))
    show_image(test_image_q, ax=ax, title="Test image: Q (128×128)")
    display(figure_to_image(fig))


def single_image():
    """Simulate a single particle image at a chosen angle and noise level."""
    angle_slider = IntSlider(value=30, min=0, max=359, step=1,
                             description="Angle (°)", style=label_style, layout=slider_layout)
    noise_sigma_slider = FloatSlider(value=1.0, min=0.0, max=8.0, step=0.5,
                                     description="Noise σ", style=label_style, layout=slider_layout)
    output_area = Output()

    def update_plot(_=None):
        simulated_image = simulate_single_image(test_image_q, angle_slider.value, noise_sigma_slider.value)
        fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
        show_image(test_image_q, axes[0], title="True structure A")
        show_image(nd_rotate(test_image_q, angle_slider.value, reshape=False),
                   axes[1], title=f"Rotated: {angle_slider.value}°")
        vmax = abs(test_image_q).max() * 1.2
        show_image(simulated_image, axes[2], vmin=-vmax, vmax=vmax,
                   title=f"Observed: θ={angle_slider.value}°, σ={noise_sigma_slider.value:.1f}")
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    for widget in [angle_slider, noise_sigma_slider]:
        widget.observe(update_plot, names='value')
    display(VBox([angle_slider, noise_sigma_slider, output_area]))
    update_plot()


def reference_selection():
    """Show how the choice of initial reference affects angular discrimination for one noisy image."""
    reference_dropdown = Dropdown(options=list(reference_images.keys()),
                                  value='Blurred Q (σ=5)',
                                  description="Reference:", style=label_style)
    noise_sigma_slider = FloatSlider(value=2.0, min=0.0, max=8.0, step=0.5,
                                     description="Noise σ", style=label_style, layout=slider_layout_wide)
    output_area = Output()

    def update_plot(_=None):
        # Always use the same particle image (angle = 130°, seed = 3) so only the reference changes
        particle_image = simulate_single_image(test_image_q, 130, noise_sigma_slider.value,
                                               np.random.default_rng(3))
        chosen_reference = reference_images[reference_dropdown.value]
        chosen_stack     = reference_stacks_5deg[reference_dropdown.value]
        alignment_scores = compute_cross_correlation(particle_image, chosen_stack)
        best_angle       = candidate_angles_5deg[alignment_scores.argmax()]

        fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
        show_image(chosen_reference, axes[0], title=f"Reference\n({reference_dropdown.value})")
        vmax = max(abs(particle_image.min()), abs(particle_image.max()))
        show_image(particle_image, axes[1], vmin=-vmax, vmax=vmax,
                   title=f"Noisy image (σ={noise_sigma_slider.value:.1f})\ntrue: 130°")
        # Alignment score curve — the peak should land near 130° for a good reference
        axes[2].plot(candidate_angles_5deg, alignment_scores, color='steelblue', lw=1.4)
        axes[2].axvline(130,        color='red',    lw=1.4, ls='--', label='True 130°')
        axes[2].axvline(best_angle, color='orange', lw=1.4, ls=':',  label=f'Best {best_angle}°')
        axes[2].set_xlabel("Candidate angle (°)")
        axes[2].set_ylabel("Correlation")
        axes[2].set_title("Alignment score")
        axes[2].legend(fontsize=8)
        axes[2].grid(alpha=0.3)
        show_image(nd_rotate(particle_image, -best_angle, reshape=False),
                   axes[3], title=f"Back-rotated {best_angle}°")
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    for widget in [reference_dropdown, noise_sigma_slider]:
        widget.observe(update_plot, names='value')
    display(VBox([reference_dropdown, noise_sigma_slider, output_area]))
    update_plot()


def alignment_demo():
    """Show the scatter of estimated vs true angles for the pre-computed example dataset."""
    angular_errors = np.abs(((estimated_angles_example - true_angles_example + 180) % 360) - 180)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].scatter(true_angles_example, estimated_angles_example, s=2, c='steelblue', alpha=0.6)
    axes[0].plot([0, 360], [0, 360], 'r--', lw=1)  # perfect-alignment diagonal
    axes[0].set_xlabel("True angle (°)")
    axes[0].set_ylabel("Estimated angle (°)")
    axes[0].set_title("Angle estimation (N=200, σ=2)")
    axes[0].set_xlim(0, 360)
    axes[0].set_ylim(0, 360)
    axes[1].hist(angular_errors, bins=36, color='steelblue', edgecolor='k', lw=0.5)
    axes[1].set_xlabel("Angular error (°)")
    axes[1].set_ylabel("Count")
    axes[1].set_title(f"Median error: {np.median(angular_errors):.1f}°")
    plt.tight_layout()
    display(figure_to_image(fig))


def reconstruction_result():
    """Show the 2-D reconstruction obtained from the pre-computed example dataset."""
    reconstructed_image = average_aligned_images(noisy_images_example, estimated_angles_example)
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
    show_image(test_image_q,            axes[0], title="True structure A")
    show_image(noisy_images_example[0], axes[1],
               vmin=noisy_images_example[0].min(),
               vmax=noisy_images_example[0].max(),
               title="Single image (σ=2.0)")
    show_image(reconstructed_image, axes[2], title="Reconstruction (N=200)")
    plt.tight_layout()
    display(figure_to_image(fig))


def reconstruction_explorer():
    """Interactively tune the number of images, noise level, and angular sampling and see the reconstruction."""
    num_images_slider    = IntSlider(  value=100, min=10,  max=500,  step=10,
                                       description="# images",   style=label_style, layout=slider_layout_wider)
    noise_sigma_slider   = FloatSlider(value=2.0, min=0.0, max=10.0, step=0.5,
                                       description="Noise σ",    style=label_style, layout=slider_layout_wider)
    blur_sigma_slider    = FloatSlider(value=5.0, min=0.0, max=15.0, step=1.0,
                                       description="LP blur σ",  style=label_style, layout=slider_layout_wider)
    angular_step_slider  = IntSlider(  value=5,   min=1,   max=20,   step=1,
                                       description="Δangle (°)", style=label_style, layout=slider_layout_wider)
    output_area = Output()

    def update_plot(_=None):
        true_angles, image_stack = simulate_particle_stack(
            test_image_q, noise_sigma_slider.value, num_images_slider.value, seed=42
        )
        # Apply a low-pass blur to the reference to reduce reference bias
        reference = (low_pass_filter(test_image_q, sigma=blur_sigma_slider.value)
                     if blur_sigma_slider.value > 0 else test_image_q.copy())
        candidate_angles, reference_stack = build_reference_stack(reference, angular_step_slider.value)
        estimated_angles  = align_particle_stack(image_stack, reference_stack, candidate_angles)
        reconstruction    = average_aligned_images(image_stack, estimated_angles)
        # Quality metric: cross-correlation between reconstruction and true structure
        quality_cc = float(
            (reconstruction * test_image_q).mean()
            / (np.sqrt((reconstruction ** 2).mean() * (test_image_q ** 2).mean()) + 1e-10)
        )
        angular_errors = np.abs(((estimated_angles - true_angles + 180) % 360) - 180)

        fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
        show_image(reference,      axes[0], title="Initial reference")
        show_image(reconstruction, axes[1], title=f"Reconstruction (CC={quality_cc:.2f})")
        axes[2].scatter(true_angles, estimated_angles, s=1, c='steelblue', alpha=0.5)
        axes[2].plot([0, 360], [0, 360], 'r--', lw=1)
        axes[2].set_xlabel("True angle (°)")
        axes[2].set_ylabel("Estimated angle (°)")
        axes[2].set_title(f"Accuracy (med. err={np.median(angular_errors):.0f}°)")
        axes[3].hist(angular_errors, bins=36, color='steelblue', ec='k', lw=0.4)
        axes[3].set_xlabel("Error (°)")
        axes[3].set_ylabel("Count")
        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    for widget in [num_images_slider, noise_sigma_slider, blur_sigma_slider, angular_step_slider]:
        widget.observe(update_plot, names='value')
    display(VBox([num_images_slider, noise_sigma_slider, blur_sigma_slider, angular_step_slider, output_area]))
    update_plot()


def translation_demo():
    """Show a gallery of simulated images with both random rotation and random translation."""
    image_stack, true_angles, true_dx, true_dy = simulate_stack_with_translation(
        test_image_q, sigma=1.0, num_images=6, max_shift=15, seed=7
    )
    fig, axes = plt.subplots(1, 7, figsize=(16, 2.8))
    show_image(test_image_q, axes[0], title="True A")
    for i in range(6):
        axes[i + 1].imshow(image_stack[i], cmap='gray', origin='lower')
        axes[i + 1].set_title(
            f"θ={true_angles[i]:.0f}°\ndx={true_dx[i]}, dy={true_dy[i]}", fontsize=7
        )
        axes[i + 1].axis('off')
    plt.suptitle("Images with rotation AND translation", fontsize=9)
    plt.tight_layout()
    display(figure_to_image(fig))


def translation_alignment():
    """Demonstrate alignment and reconstruction when particles have unknown positions."""
    num_images = 80
    image_stack, true_angles, true_dx, true_dy = simulate_stack_with_translation(
        test_image_q, 1.0, num_images, max_shift=10, seed=42
    )
    # Build a reference stack that covers all (angle, dx, dy) combinations
    blurred_reference = low_pass_filter(test_image_q, sigma=5)
    reference_stack, angle_list, dx_list, dy_list = build_rotation_translation_reference_stack(
        blurred_reference, angular_step_deg=10, max_shift=10, shift_step=4
    )
    estimated_angles, estimated_dx, estimated_dy = align_with_translation(
        image_stack, reference_stack, angle_list, dx_list, dy_list
    )
    reconstruction = reconstruct_with_translation(image_stack, estimated_angles, estimated_dx, estimated_dy)

    fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
    show_image(test_image_q,      axes[0], title="True A")
    show_image(image_stack[0],    axes[1],
               vmin=image_stack[0].min(), vmax=image_stack[0].max(),
               title="Sample image (σ=1.0)")
    show_image(reconstruction, axes[2], title=f"Reconstruction with translation (N={num_images})")
    plt.tight_layout()
    display(figure_to_image(fig))


def ml_comparison():
    """Compare hard-assignment and ML-soft reconstruction at different noise levels."""
    reference_dropdown = Dropdown(options=list(ml_reference_images.keys()),
                                  value='Blurred Q', description='Reference:', style=label_style)
    true_angle = 37   # fixed ground-truth angle for the single-image likelihood plots
    noise_levels = [1.5, 3.0, 6.0]
    reconstruction_cache = {}   # avoid recomputing when only the display reference changes
    output_area = Output()

    def update_plot(_=None):
        reference_name  = reference_dropdown.value
        reference_stack = ml_reference_stacks[reference_name]
        reference_image = ml_reference_images[reference_name]

        if reference_name not in reconstruction_cache:
            # Pre-compute hard and ML reconstructions for all noise levels
            candidate_angles_coarse, rotated_refs_coarse = build_reference_stack(reference_image, angular_step_deg=15)
            reconstruction_cache[reference_name] = {}
            for sigma in noise_levels:
                image_set = ml_noisy_images[sigma]
                estimated = align_particle_stack(image_set, rotated_refs_coarse, candidate_angles_coarse)
                hard_reconstruction = average_aligned_images(image_set, estimated)
                soft_reconstruction = ml_reconstruct(image_set, rotated_refs_coarse,
                                                     candidate_angles_coarse, noise_sigma=min(sigma, 2.0))
                reconstruction_cache[reference_name][sigma] = (hard_reconstruction, soft_reconstruction)

        fig, axes = plt.subplots(3, 4, figsize=(16, 10),
                                 gridspec_kw={'width_ratios': [1, 1, 1, 0.7]})
        # Column 4: show the reference image
        axes[0, 3].imshow(reference_image, cmap='gray', origin='lower')
        axes[0, 3].axis('off')
        axes[0, 3].set_title('Reference', fontsize=9)
        axes[1, 3].axis('off')
        axes[2, 3].axis('off')

        for col, sigma in enumerate(noise_levels):
            # Row 0: ML weight distribution for a single example image
            example_image = simulate_single_image(test_image_q, true_angle, sigma, np.random.default_rng(42))
            residuals = reference_stack - example_image[None]
            log_likelihoods = -0.5 * np.sum(residuals ** 2, axis=(1, 2)) / max(sigma ** 2, 0.25)
            log_likelihoods -= log_likelihoods.max()
            weights = np.exp(log_likelihoods)
            weights /= weights.sum()
            best_hard_angle = ml_candidate_angles[weights.argmax()]

            axes[0, col].bar(ml_candidate_angles, weights, width=8, color='steelblue', alpha=0.8)
            axes[0, col].axvline(true_angle,      color='red',    lw=1.5, ls='--', label=f'True {true_angle}°')
            axes[0, col].axvline(best_hard_angle, color='orange', lw=1.5, ls=':',  label=f'Hard {best_hard_angle}°')
            axes[0, col].set_xlim(0, 360)
            axes[0, col].set_xlabel('Angle (°)', fontsize=7)
            axes[0, col].set_yticks([])
            axes[0, col].tick_params(labelsize=7)
            axes[0, col].set_title(f'σ={sigma}', fontsize=9)
            if col == 0:
                axes[0, col].legend(fontsize=6)

            hard_reconstruction, soft_reconstruction = reconstruction_cache[reference_name][sigma]
            axes[1, col].imshow(hard_reconstruction, cmap='gray', origin='lower')
            axes[1, col].axis('off')
            axes[1, col].set_title(f'Hard (σ={sigma})', fontsize=9)
            axes[2, col].imshow(soft_reconstruction, cmap='gray', origin='lower')
            axes[2, col].axis('off')
            axes[2, col].set_title(f'ML soft (σ={sigma})', fontsize=9)

        axes[0, 0].set_ylabel('ML weights $w_{ij}$', fontsize=9)
        axes[1, 0].set_ylabel('Hard recon',          fontsize=9)
        axes[2, 0].set_ylabel('ML soft recon',       fontsize=9)
        plt.suptitle(
            f'Reference: {reference_name}  |  true angle = {true_angle}°  |  10 images/noise level',
            fontsize=10
        )
        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    reference_dropdown.observe(update_plot, names='value')
    display(VBox([reference_dropdown, output_area]))
    update_plot()


def iterative_refinement():
    """Run iterative alignment: use the current reconstruction as the reference for the next round."""
    num_iterations_slider = IntSlider(  value=3,   min=1,   max=8,    step=1,
                                        description="Iterations",   style=label_style, layout=slider_layout_wider)
    blur_sigma_slider     = FloatSlider(value=5.0, min=0.0, max=15.0, step=1.0,
                                        description="Init. blur σ", style=label_style, layout=slider_layout_wider)
    noise_sigma_slider    = FloatSlider(value=2.0, min=0.0, max=10.0, step=0.5,
                                        description="Noise σ",      style=label_style, layout=slider_layout_wider)
    start_reference_dropdown = Dropdown(
        options=['Q (oracle)', 'Blurred Q', 'Circle', 'A (wrong)', 'P (wrong)', 'O (wrong)'],
        value='Blurred Q',
        description="Start ref:", style=label_style
    )
    output_area = Output()

    def update_plot(_=None):
        true_angles, image_stack = simulate_particle_stack(test_image_q, noise_sigma_slider.value, 100, seed=42)
        reference_name = start_reference_dropdown.value
        blur_sigma     = blur_sigma_slider.value

        # Select the initial reference based on the dropdown choice
        if reference_name == 'Q (oracle)':
            current_reference = test_image_q.copy()
        elif reference_name == 'Circle':
            current_reference = make_circle_image(128)
        elif reference_name == 'Blurred Q':
            current_reference = low_pass_filter(test_image_q, sigma=blur_sigma)
        else:
            letter = reference_name.split(' ')[0]
            letter_image = make_letter_image(letter, 128)
            current_reference = (low_pass_filter(letter_image, sigma=blur_sigma)
                                 if blur_sigma > 0 else letter_image)

        # Each iteration: align to current reference, then reconstruct a new reference
        reference_history = [current_reference.copy()]
        quality_scores = []
        for _ in range(num_iterations_slider.value):
            candidate_angles, reference_stack = build_reference_stack(current_reference, angular_step_deg=5)
            estimated_angles   = align_particle_stack(image_stack, reference_stack, candidate_angles)
            current_reference  = average_aligned_images(image_stack, estimated_angles)
            reference_history.append(current_reference.copy())
            quality_cc = float(
                (current_reference * test_image_q).mean()
                / (np.sqrt((current_reference ** 2).mean() * (test_image_q ** 2).mean()) + 1e-10)
            )
            quality_scores.append(quality_cc)

        num_cols = min(len(reference_history), 6)
        fig, axes = plt.subplots(1, num_cols + 1, figsize=(3 * (num_cols + 1), 3.5))
        show_image(test_image_q, axes[0], title="True A")
        labels = ['Init'] + [f'Iter {i + 1}' for i in range(num_cols - 1)]
        for col, (image, label) in enumerate(zip(reference_history[:num_cols], labels), start=1):
            axes[col].imshow(image, cmap='gray', origin='lower')
            axes[col].axis('off')
            cc_label = '' if col == 1 else f'\nCC={quality_scores[col - 2]:.2f}'
            axes[col].set_title(label + cc_label, fontsize=8)
        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    for widget in [num_iterations_slider, blur_sigma_slider, noise_sigma_slider, start_reference_dropdown]:
        widget.observe(update_plot, names='value')
    display(VBox([start_reference_dropdown, blur_sigma_slider, noise_sigma_slider,
                  num_iterations_slider, output_area]))
    update_plot()


def classification():
    """Classify a mixed dataset of K letter classes and reconstruct each class separately."""
    num_classes_slider      = IntSlider(  value=2,   min=2,   max=3,   step=1,
                                          description="# classes K",   style=label_style, layout=slider_layout_wider)
    images_per_class_slider = IntSlider(  value=40,  min=10,  max=100, step=5,
                                          description="Images/class",  style=label_style, layout=slider_layout_wider)
    noise_sigma_slider      = FloatSlider(value=1.5, min=0.0, max=6.0, step=0.5,
                                          description="Noise σ",       style=label_style, layout=slider_layout_wider)
    output_area = Output()

    def update_plot(_=None):
        num_classes      = num_classes_slider.value
        num_per_class    = images_per_class_slider.value
        noise_sigma      = noise_sigma_slider.value
        active_letters   = class_letters[:num_classes]
        rng              = np.random.default_rng(42)

        # Generate images from each class at random angles, then shuffle
        all_images, true_labels = [], []
        for class_index, letter in enumerate(active_letters):
            angle_indices = rng.integers(0, len(class_candidate_angles), num_per_class)
            noisy_images = (class_reference_stacks[letter][angle_indices]
                            + rng.standard_normal((num_per_class, 128, 128)) * noise_sigma)
            all_images.append(noisy_images)
            true_labels.extend([class_index] * num_per_class)
        all_images   = np.concatenate(all_images)
        true_labels  = np.array(true_labels)
        shuffle_order = rng.permutation(len(all_images))
        all_images   = all_images[shuffle_order]
        true_labels  = true_labels[shuffle_order]

        # Build one big reference stack covering all classes and all angles
        combined_reference_stack = np.concatenate([class_reference_stacks[lt] for lt in active_letters])
        num_angles_per_class = len(class_candidate_angles)
        num_all_images       = all_images.shape[0]

        # Fast matrix cross-correlation: shape (num_images, num_references)
        images_flat     = all_images.reshape(num_all_images, 128 * 128).astype(float)
        references_flat = combined_reference_stack.reshape(len(combined_reference_stack), 128 * 128).astype(float)
        correlation_matrix  = (images_flat @ references_flat.T) / (128 * 128)
        best_reference_indices = correlation_matrix.argmax(1)
        # The class index and the within-class angle index follow from the combined stack layout
        predicted_classes  = best_reference_indices // num_angles_per_class
        best_angle_indices = best_reference_indices % num_angles_per_class
        classification_accuracy = (predicted_classes == true_labels).mean() * 100

        # Reconstruct each class by averaging back-rotated images assigned to it
        class_reconstructions = []
        for class_index in range(num_classes):
            class_mask    = (predicted_classes == class_index)
            class_recon   = np.zeros((128, 128))
            num_in_class  = 0
            for image_index in np.where(class_mask)[0]:
                angle = class_candidate_angles[best_angle_indices[image_index]]
                class_recon += nd_rotate(all_images[image_index], -angle, reshape=False)
                num_in_class += 1
            class_reconstructions.append(class_recon / max(num_in_class, 1))

        fig, axes = plt.subplots(1, num_classes + 2, figsize=(3.5 * (num_classes + 2), 3.5))
        show_image(all_images.mean(0), axes[0], title="Naive average\n(blurry)")
        for class_index, (recon, letter) in enumerate(zip(class_reconstructions, active_letters)):
            axes[class_index + 1].imshow(recon, cmap='gray', origin='lower')
            axes[class_index + 1].axis('off')
            axes[class_index + 1].set_title(
                f"Class {class_index + 1}: {letter}\n"
                f"({(predicted_classes == class_index).sum()} imgs, acc={classification_accuracy:.0f}%)",
                fontsize=8
            )
        axes[num_classes + 1].axis('off')
        plt.suptitle(
            f"2D Classification: K={num_classes}, σ={noise_sigma}, {num_per_class} imgs/class",
            fontsize=9
        )
        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    for widget in [num_classes_slider, images_per_class_slider, noise_sigma_slider]:
        widget.observe(update_plot, names='value')
    display(VBox([num_classes_slider, images_per_class_slider, noise_sigma_slider, output_area]))
    update_plot()


print("Setup complete.")


# Practical: Single-Particle Analysis

This practical covers the core algorithmic steps of single-particle analysis: alignment, reconstruction, and classification.

> **To start the practicals:** click **Run All** (⏭) in the toolbar above.

***

## Part 0 – The test image

In [ ]:
show_test_image()

***

## Simulating cryo-EM images

The interactive below shows a single simulated particle image. The image formation model is:

$$
X = R^\theta A + \sigma G
$$

Drag the **angle** slider to rotate the particle and the **σ** slider to add noise.

In [ ]:
single_image()

***

## Part 1 – Choosing an initial reference

The alignment algorithm needs a starting reference to compare images against. Three strategies are common:

| Reference | Advantage | Risk |
|-----------|-----------|------|
| Low-pass filtered true structure | Fast convergence | Only works if structure is already known (circular reasoning) |
| Featureless circle | No reference bias | Slow convergence; poor angular discrimination |
| Random noise / another structure | Quick to obtain | Risk of converging to wrong answer (model bias) |

In practice, a featureless circle or a low-pass-filtered version of a prior reconstruction is used. The interactive below lets you choose the starting reference and see how much angular discrimination it provides for a single noisy image.

In [ ]:
reference_selection()

***

## Part 2 – Alignment algorithm

For a set of $N$ images $\{X_i\}$, we align each to the reference by searching over discrete candidate angles $\Delta\theta$ apart. For each image we rotate the reference to all candidate angles and pick the best-matching one:

$$
\hat\theta_i = \arg\max_{\theta_j} \bigl(X_i \cdot R^{\theta_j} A_\text{ref}\bigr)
$$

After estimating all angles, the scatter plot of true vs estimated angles reveals how well alignment worked. A tight diagonal indicates good alignment; spread indicates confusion.

```{admonition} Task
:class: note
**Part 2, step 1:** Implement the alignment loop in the code cell below. For each image, compute the correlation against every rotated reference and record the best angle. Run the cell to see the scatter plot of true vs estimated angles.
```

In [ ]:
alignment_demo()

***

## Part 3 – Reconstruction

Given estimated angles, rotating each image back and averaging gives the reconstruction:

$$
A \leftarrow \frac{1}{N} \sum_{i=1}^N (R^{\hat\theta_i})^{-1} X_i
$$

```{admonition} Task
:class: note
**Part 3:** Complete the reconstruction: for each image, rotate it back by $-\hat\theta_i$ and sum all rotated images. Divide by $N$ to get the average.
```

In [ ]:
reconstruction_result()

***

## Interactive reconstruction

The widget below runs the complete pipeline — from noisy images to reconstruction — with adjustable parameters. It also shows the scatter of estimated vs true angles and the correlation between reconstruction and true image as a quality metric.

In [ ]:
reconstruction_explorer()

***

## Bonus 1 – Adding translation

So far we assumed particles are perfectly centred. In reality, particle positions within the extracted box vary. The model extends to:

$$
X_i = T^{t_i}(R^{\theta_i} A) + \sigma G_i
$$

where $T^{t_i}$ is a translation by vector $\mathbf{t}_i = (dx_i, dy_i)$. We can handle integer-pixel translations by rolling the image array. The search grid is now $N_\theta \times N_{dx} \times N_{dy}$, so runtime increases by a factor of $(2 t_\text{max}+1)^2$.

In [ ]:
translation_demo()

In [ ]:
translation_alignment()

***

## Bonus 2 – Bayesian maximum-likelihood alignment

Hard assignment to the single best-fitting angle ignores alignment uncertainty. The **maximum-likelihood** approach instead computes a probability for each angle:

$$
P(X_i \mid \theta_j, A) \propto \exp\!\left[-\frac{\|X_i - R^{\theta_j}A\|^2}{2\sigma^2}\right]
$$

The **soft** reconstruction then uses all angles weighted by their posterior:

$$
w_{ij} = \frac{P(X_i \mid \theta_j, A)}{\sum_{j'} P(X_i \mid \theta_{j'}, A)}, \qquad
A \leftarrow \sum_i \sum_j w_{ij} (R^{\theta_j})^{-1} X_i
$$

At high SNR the weights collapse onto the single best angle (recovering hard assignment); at low SNR they spread across many angles, effectively averaging out alignment noise.

In [ ]:
ml_comparison()

***

## Bonus 3 – Iterative refinement

The previous examples used the initial reference for all alignment. A better strategy is to use the reconstruction from one round as the reference for the next:

1. Start from initial reference $A^{(0)}$ (e.g. blurred Q or a circle)
2. Align images to $A^{(k)}$ → estimate $\hat\theta_i^{(k)}$
3. Reconstruct $A^{(k+1)}$ from aligned images
4. Repeat from step 2

The quality of the reconstruction typically improves each iteration. The widget below lets you choose the starting reference and the number of iterations.

In [ ]:
iterative_refinement()

***

## Bonus 4 – 2D Classification

Real datasets contain multiple particle types (different proteins, conformations, or viewing directions). **2D classification** assigns each image to one of $K$ classes simultaneously with the alignment:

1. Maintain $K$ references $\{A_1, \ldots, A_K\}$
2. For each image, find the best (class, angle) pair by correlation
3. Reconstruct each class from the images assigned to it
4. Update references and iterate

The interactive below mixes $K$ letters (Q, P, R) and attempts to recover them. The classification accuracy depends on the noise level and the number of images per class.

In [ ]:
classification()

***

## Summary

| Topic | Key concept |
|-------|-------------|
| Observation model | $X_i = R^{\theta_i}A + \sigma G_i$ |
| Reference selection | Circular, blurred prior, or oracle; avoid reference bias |
| Alignment | Maximise correlation $\langle X_i, R^{\theta_j}A\rangle$ over discrete candidates |
| Reconstruction | Average back-rotated images |
| Model bias | Wrong reference → biased reconstruction; correct with iterative refinement |
| ML alignment | Soft weights via $\exp(-\|X_i - R^\theta A\|^2/2\sigma^2)$; better at low SNR |
| Iterative refinement | Use reconstruction as new reference; converges toward true structure |
| Classification | $K$-class joint alignment+assignment; separates heterogeneous datasets |